# qwen3.5 audit v11 right tool rubric clip eval 1fps

This notebook adapts the right-tool rubric prompting structure from
`notebooks/right/qwen3.5_coarse_start_mid_end_exact_question_eval_pm1s_b.ipynb`
to audit-v11 clips.

- It keeps the same rubric questions, same prompt wording, and same rubric-to-tool aggregation logic.
- It changes the image bundle so the prompt sees the whole clip instead of a single target frame.
- It predicts the right-hand tool only and compares clip-level tool accuracy with
  `structured_prediction_cvs_context` results from `qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3.ipynb`.

Frame sampling for this notebook: `all_images_1fps`.
Artifacts/cache for this notebook are written under `notebooks/qwen3.5_audit_v11_seg_extract_spec_eval/artifacts/qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps*`.


In [1]:
from openai import OpenAI
import base64
import io
import json
import hashlib
import os
import re
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from PIL import Image
from IPython.display import display

with open('/mnt/md0/weiqiuy/ips/carnaroli.txt') as input_file:
    carnaroli_ip = input_file.read().strip()

client = OpenAI(
    base_url=f"http://{carnaroli_ip}:8001/v1",
    api_key=os.environ['OPENAI_API_KEY'],
)

MODEL_ID = "Qwen/Qwen3.6-35B-A3B-FP8"
ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
FRAMES_DIR = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/frames/test'
GT_PATH = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5/audit_v11_simple_actions.json'
COMPARE_RESULTS_PATH = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5/results.json'
ARTIFACT_DIR = ROOT_DIR / 'notebooks/qwen3.5_audit_v11_seg_extract_spec_eval/artifacts'
CACHE_PATH = ARTIFACT_DIR / 'qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps_cache.json'

FRAME_SAMPLING_LABEL = 'all_images_1fps'
FRAME_STRIDE = 30

print('Model:', MODEL_ID)
print('Frames dir:', FRAMES_DIR)
print('GT path:', GT_PATH)
print('Compare results:', COMPARE_RESULTS_PATH)
print('Cache path:', CACHE_PATH)
print('Frame sampling:', FRAME_SAMPLING_LABEL, 'stride=', FRAME_STRIDE)


Model: Qwen/Qwen3.6-35B-A3B-FP8
Frames dir: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/frames/test
GT path: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5/audit_v11_simple_actions.json
Compare results: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5/results.json
Cache path: /mnt/md0/weiqiuy/surgent/notebooks/qwen3.5_audit_v11_seg_extract_spec_eval/artifacts/qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps_cache.json
Frame sampling: all_images_1fps stride= 30


In [2]:
REFERENCE_IMAGE_PATHS = {
    "holes": ROOT_DIR / "few_shot_examples/shape/2d/holes_vs_no_holes.png",
    "sharp": ROOT_DIR / "few_shot_examples/shape/2d/sharp_vs_blunt.png",
    "tapered": ROOT_DIR / "few_shot_examples/shape/2d/tapered_vs_constant.png",
    "tube": ROOT_DIR / "few_shot_examples/shape/2d/tube_vs_with_tiip.png",
    "two_tips": ROOT_DIR / "few_shot_examples/shape/2d/two_tips_vs_one.png",
    "wire": ROOT_DIR / "few_shot_examples/shape/2d/wire_vs_not_wire.png",
    "hollow": ROOT_DIR / "few_shot_examples/shape/2d/hollow_vs_solid_tip.png",
    "hinge": ROOT_DIR / "few_shot_examples/shape/2d/hinge_vs_no_hinge.png",
    "serration": ROOT_DIR / "few_shot_examples/shape/2d/serration_vs_smooth.png",
    "pivot": ROOT_DIR / "few_shot_examples/shape/2d/pivot_vs_parallel.png",
}

instrument_rubric = [
    {
        "id": 1,
        "question": "Does the distal tube have hole-like openings?",
        "description": {
            "yes": "Yes if there are one or more visible holes on the tube/shaft. Note that this should only be holes on a metal tube with metal color instead of other textured tubes. And the holes should be small round holes on the tube instead of wide rectangle openings without closure at the end of the tool.",
            "no": "No if the tube/shaft surface does not have holes. If the tool is not metal, or if there is blood that looks like round holes, then those are not holes.",
            "uncertain": "Uncertain if the image is blurry or if the end of the tube/shaft is obscured by tissue or other tools. If there is no tool operating, then it should also be uncertain.",
        },
        "reference_name": "holes",
        "reference_description": "Reference image for distinguishing visible hole-like openings on a tube from a smooth tube.",
        "scores": {
            "Grasper": ("No", 2),
            "Maryland": ("No", 2),
            "Hook": ("No", 2),
            "Irrigator": ("Yes", 2),
            "Scissors": ("No", 2),
            "Clipper": ("No", 2),
        },
    },
    {
        "id": 2,
        "question": "Does the tip (each tip, if multiple) have a cutting edge (blade)?",
        "description": {
            "yes": "Yes if sharp cutting edges or blades are visible for the metal tips of the tool. We only consider blades between two metal tips, on the inside where two metal tips are adjacent to each other, instead of blades outside. Do not choose yes if the tips are buried in tissue and cannot be confirmed.",
            "no": "No if the tips of the tool are blunt. Do not choose no if the tips are buried in tissue and cannot be confirmed.",
            "uncertain": "Uncertain if the image is blurry or if the tips of the tool are obscured by tissue or other tools. If there is no tool operating, then it should also be uncertain.",
        },
        "reference_name": "sharp",
        "reference_description": "Reference image for distinguishing a sharp cutting edge from a non-bladed tip.",
        "scores": {
            "Grasper": ("No", 2),
            "Maryland": ("No", 2),
            "Hook": (None, 0),
            "Irrigator": ("No", 2),
            "Scissors": ("Yes", 2),
            "Clipper": ("No", 2),
        },
    },
    {
        "id": 3,
        "question": "Is the metal tip of the tool (each tip, if multiple) tapered?",
        "description": {
            "yes": "Yes if the width decreases smoothly toward the tip (tapered).",
            "no": "No if the width is mostly constant or only slightly changes (not tapered).",
            "uncertain": "Uncertain if the image is blurry or if the tips of the tool are obscured by tissue or other tools. If there is no tool operating, then it should also be uncertain.",
        },
        "reference_name": "tapered",
        "reference_description": "Reference image for distinguishing a tapered tip from a non-tapered tip.",
        "scores": {
            "Grasper": ("No", 2),
            "Maryland": ("Yes", 2),
            "Hook": ("No", 2),
            "Irrigator": ("No", 2),
            "Scissors": ("Yes", 2),
            "Clipper": (None, 0),
        },
    },
    {
        "id": 4,
        "question": "Does the distal end consist of a tube only (without any separate tip structure)?",
        "description": {
            "yes": "Yes if the end is just a tube.",
            "no": "No if there is a distinct tip or multiple tips attached.",
            "uncertain": "Uncertain if the distal end is blurry or obscured by tissue or other tools, or if there is no clearly operating tool.",
        },
        "reference_name": "tube",
        "reference_description": "Reference image for distinguishing a tube-only distal end from one with a distinct tip structure.",
        "scores": {
            "Grasper": ("No", 2),
            "Maryland": ("No", 2),
            "Hook": ("No", 2),
            "Irrigator": ("Yes", 2),
            "Scissors": ("No", 2),
            "Clipper": ("No", 2),
        },
    },
    {
        "id": 5,
        "question": "Does the tip split into two visible parts?",
        "description": {
            "yes": "Yes if two separate tips or jaws are visible.",
            "no": "No if only a single continuous tip is visible.",
            "uncertain": "Uncertain if the tip is blurry or obscured by tissue or other tools, or if there is no clearly operating tool.",
        },
        "reference_name": "two_tips",
        "reference_description": "Reference image for distinguishing two visible tips or jaws from a single continuous tip.",
        "scores": {
            "Grasper": ("Yes", 1),
            "Maryland": ("Yes", 1),
            "Hook": ("No", 1),
            "Irrigator": ("No", 1),
            "Scissors": ("Yes", 1),
            "Clipper": ("Yes", 1),
        },
    },
    {
        "id": 6,
        "question": "Does the tip appear wire-like (very thin and rod-like, much thinner than typical jaws or blades)?",
        "description": {
            "yes": "Yes if the tip resembles a thin metal wire.",
            "no": "No if the tip is flat, broad, or has noticeable thickness.",
            "uncertain": "Uncertain if the tip is blurry or obscured by tissue or other tools, or if there is no clearly operating tool.",
        },
        "reference_name": "wire",
        "reference_description": "Reference image for distinguishing a wire-like tip from a flat, broad, or thick tip.",
        "scores": {
            "Grasper": ("No", 2),
            "Maryland": ("No", 2),
            "Hook": ("Yes", 2),
            "Irrigator": ("No", 2),
            "Scissors": ("No", 2),
            "Clipper": ("No", 2),
        },
    },
    {
        "id": 7,
        "question": "Does the tip appear hollow (having an opening or lumen) rather than solid?",
        "description": {
            "yes": "Yes if an opening or inner cavity is visible.",
            "no": "No if the tip appears solid with no opening.",
            "uncertain": "Uncertain if the tip is blurry or obscured by tissue or other tools, or if there is no clearly operating tool.",
        },
        "reference_name": "hollow",
        "reference_description": "Reference image for distinguishing a hollow tip from a solid tip.",
        "scores": {
            "Grasper": ("Yes", 2),
            "Maryland": ("No", 2),
            "Hook": ("No", 2),
            "Irrigator": ("No", 2),
            "Scissors": ("No", 2),
            "Clipper": ("No", 2),
        },
    },
    {
        "id": 8,
        "question": "Is a hinge (pivot joint) visible where the tip is attached to the shaft?",
        "description": {
            "yes": "Yes if a pivot, pin, or joint structure is visible between the tip and where it connects to the shaft.",
            "no": "No if no such joint is visible.",
            "uncertain": "Uncertain if the hinge region is blurry or obscured by tissue or other tools, or if there is no clearly operating tool.",
        },
        "reference_name": "hinge",
        "reference_description": "Reference image for distinguishing a visible hinge or pivot from no hinge.",
        "scores": {
            "Grasper": ("Yes", 1),
            "Maryland": ("Yes", 1),
            "Hook": ("No", 1),
            "Irrigator": ("No", 1),
            "Scissors": ("Yes", 1),
            "Clipper": ("Yes", 1),
        },
    },
    {
        "id": 9,
        "question": "Do any visible tips have serrations (small, repeated tooth-like ridges along the edge)?",
        "description": {
            "yes": "Yes if a saw or comb-like pattern is visible.",
            "no": "No if the edge is smooth or flat.",
            "uncertain": "Uncertain if the edge is blurry or obscured by tissue or other tools, or if there is no clearly operating tool.",
        },
        "reference_name": "serration",
        "reference_description": "Reference image for distinguishing serrated edges from smooth edges.",
        "scores": {
            "Grasper": ("Yes", 1),
            "Maryland": ("Yes", 2),
            "Hook": ("No", 2),
            "Irrigator": ("No", 2),
            "Scissors": ("No", 2),
            "Clipper": ("No", 2),
        },
    },
    {
        "id": 10,
        "question": "Do the two tips originate from a common pivot point at the base (opening like a fan)?",
        "description": {
            "yes": "Yes if both tips meet at a single joint and spread apart from that point.",
            "no": "No if the tips remain parallel and do not converge to a single pivot point.",
            "uncertain": "Uncertain if the base of the two tips is blurry or obscured by tissue or other tools, or if there is no clearly operating tool.",
        },
        "reference_name": "pivot",
        "reference_description": "Reference image for distinguishing tips that fan open from a common pivot versus tips that remain parallel.",
        "scores": {
            "Grasper": ("Yes", 2),
            "Maryland": ("Yes", 2),
            "Hook": (None, 0),
            "Irrigator": (None, 0),
            "Scissors": ("Yes", 2),
            "Clipper": ("No", 2),
        },
    },
]

frame_checks = [
    {
        "id": 1,
        "question": "Is the active working instrument (the one performing dissection, cutting, aspiration, or similar, not the instrument retracting the gallbladder) present in the frame?",
        "description": {
            "yes": "Yes if the active working instrument is visibly present in the frame.",
            "no": "No if the active working instrument is clearly absent from the frame.",
            "uncertain": "Uncertain if visibility is too poor or the frame does not provide enough evidence to decide whether the active working instrument is present.",
        },
        "value": 1,
    },
    {
        "id": 2,
        "question": "Is the tip of the actively working instrument obscured by tissue?",
        "description": {
            "yes": "Yes if the tip of the active working instrument is covered or blocked by tissue.",
            "no": "No if the tip of the active working instrument is not obscured by tissue.",
            "uncertain": "Uncertain if the instrument tip cannot be located clearly enough to judge whether tissue is obscuring it.",
        },
        "value": -1,
    },
    {
        "id": 3,
        "question": "Is the tip of the actively working instrument blurry?",
        "description": {
            "yes": "Yes if the tip of the active working instrument appears blurry or out of focus.",
            "no": "No if the tip of the active working instrument appears sharp enough to inspect.",
            "uncertain": "Uncertain if the instrument tip cannot be located clearly enough to judge whether it is blurry.",
        },
        "value": -1,
    },
]

TOOL_TYPES = ["Grasper", "Maryland", "Hook", "Irrigator", "Scissors", "Clipper"]
QUESTION_SPECS = []
for item in instrument_rubric:
    QUESTION_SPECS.append({
        "qid": f"instrument_{item['id']}",
        "group": "instrument",
        "question": item["question"],
        "description": item["description"],
        "reference_name": item["reference_name"],
        "reference_description": item["reference_description"],
    })
for item in frame_checks:
    QUESTION_SPECS.append({
        "qid": f"frame_check_{item['id']}",
        "group": "frame_check",
        "question": item["question"],
        "description": item["description"],
        "reference_name": None,
        "reference_description": None,
    })

print('Loaded', len(QUESTION_SPECS), 'questions')


Loaded 13 questions


In [3]:

def encode_image_file(path: Path):
    suffix = path.suffix.lower()
    mime = 'image/png' if suffix == '.png' else 'image/jpeg'
    b64 = base64.b64encode(path.read_bytes()).decode()
    return mime, b64

reference_images = {}
for name, path in REFERENCE_IMAGE_PATHS.items():
    mime, b64 = encode_image_file(path)
    reference_images[name] = {"path": str(path), "mime": mime, "b64": b64}


def normalize_answer(value):
    if value is None:
        return ''
    value = str(value).strip()
    if not value:
        return ''
    lowered = value.lower()
    mapping = {
        'yes': 'Yes',
        'no': 'No',
        'uncertain': 'Uncertain',
        'unknown': 'Unknown',
        '(absent)': '(absent)',
        'true': 'Yes',
        'false': 'No',
    }
    return mapping.get(lowered, value)


def extract_binary_answer(text):
    cleaned = (text or '').split('</think>')[-1].strip().lower()
    match = re.search(r'\b(yes|no|uncertain)\b', cleaned)
    return normalize_answer(match.group(1) if match else 'uncertain')


def extract_json_object(text):
    cleaned = (text or '').strip()
    cleaned = re.sub(r'<think>.*?</think>', '', cleaned, flags=re.DOTALL).strip()
    if cleaned.startswith('```'):
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
        cleaned = re.sub(r'\s*```$', '', cleaned)
    if not cleaned.startswith('{'):
        match = re.search(r'(\{.*\})', cleaned, re.DOTALL)
        if match:
            cleaned = match.group(1)
    return json.loads(cleaned)


def image_path_to_data_url(image_path: Path):
    mime, b64 = encode_image_file(image_path)
    return f'data:{mime};base64,{b64}'


def collect_annotation_answer_map(clip):
    row_map = {ann['row_id'].split('__')[-1]: ann for ann in clip['annotations']}
    out = {}
    for row_key, ann in row_map.items():
        out[row_key] = {
            'sequence_answer': normalize_answer(ann.get('sequence_answer', '')),
            'frame_answers': {str(k): normalize_answer(v) for k, v in ann.get('frame_answers', {}).items()},
        }
    return out


def effective_gt_answer(answer_entry, frame_id):
    if answer_entry is None:
        return ''
    frame_value = normalize_answer(answer_entry.get('frame_answers', {}).get(str(frame_id), ''))
    if frame_value:
        return frame_value
    return normalize_answer(answer_entry.get('sequence_answer', ''))


def derive_gt_tool_name(answer_map, frame_id):
    tool_answer = effective_gt_answer(answer_map.get('tool_name'), frame_id)
    if tool_answer:
        return tool_answer
    present_answer = effective_gt_answer(answer_map.get('frame_check_1'), frame_id)
    if present_answer == 'No':
        return '(absent)'
    return ''


def derive_clip_level_tool_name(clip):
    audited = clip.get('audited_right_tool_gt') or []
    audited = [normalize_answer(x) for x in audited if normalize_answer(x)]
    if audited:
        return audited[0]
    return ''



TOOL_LABEL_MAP = {
    'grasper': 'Grasper',
    'maryland': 'Maryland',
    'hook': 'Hook',
    'irrigator': 'Irrigator',
    'scissors': 'Scissors',
    'clipper': 'Clipper',
    'unknown': 'Unknown',
    '(absent)': '(absent)',
    '(not set)': '(absent)',
    'not set': '(absent)',
    'none': '(absent)',
}


def normalize_gt_tool_label(value):
    if value is None:
        return ''
    value = str(value).strip()
    if not value:
        return ''
    return TOOL_LABEL_MAP.get(value.lower(), value)


def sampled_frame_ids(frame_range, frame_stride):
    start_frame, end_frame = [int(x) for x in frame_range]
    frame_ids = list(range(start_frame, end_frame + 1, int(frame_stride)))
    if not frame_ids:
        frame_ids = [start_frame]
    if frame_ids[-1] != end_frame:
        frame_ids.append(end_frame)
    return sorted(dict.fromkeys(frame_ids))


def collect_clip_frame_paths(video_id, frame_range, frame_stride):
    frame_ids = sampled_frame_ids(frame_range, frame_stride)
    paths = []
    for frame_id in frame_ids:
        path = FRAMES_DIR / video_id / f'frame_{int(frame_id):06d}.png'
        if path.exists():
            paths.append(str(path))
    return paths


def derive_clip_gt_tool(row):
    tools = sorted({
        normalize_gt_tool_label(seg.get('tool_type'))
        for seg in row.get('right', [])
        if normalize_gt_tool_label(seg.get('tool_type'))
    })
    if not tools:
        return '(absent)', '(absent)', True
    if len(tools) == 1:
        return tools[0], tools[0], True
    return '', ' | '.join(tools), False


def build_eval_examples():
    rows = json.loads(GT_PATH.read_text())
    examples = []
    for row in rows:
        image_paths = collect_clip_frame_paths(row['video_id'], row['frame_range'], FRAME_STRIDE)
        if not image_paths:
            continue
        gt_tool_name, gt_tool_set_text, gt_is_evaluable = derive_clip_gt_tool(row)
        examples.append({
            'example_uid': row['example_id'],
            'video_id': row['video_id'],
            'clip_uid': row['example_id'],
            'criterion': row.get('criterion', ''),
            'frame_tag': FRAME_SAMPLING_LABEL,
            'frame_id': int(row['frame_range'][0]),
            'frame_range': row['frame_range'],
            'start_frame': int(row['frame_range'][0]),
            'end_frame': int(row['frame_range'][1]),
            'image_path': image_paths,
            'gt_answers': {},
            'gt_tool_name': gt_tool_name,
            'gt_tool_set_text': gt_tool_set_text,
            'clip_level_tool_name': gt_tool_name,
            'tool_matches_clip_level': bool(gt_tool_name),
            'labeled_question_count': 0,
            'n_images': len(image_paths),
            'gt_is_evaluable': gt_is_evaluable,
        })
    return examples


eval_examples = build_eval_examples()
df_eval_examples = pd.DataFrame(eval_examples)
print('Audit-v11 clip eval items:', len(eval_examples))
display(
    df_eval_examples[
        ['example_uid', 'video_id', 'criterion', 'frame_range', 'start_frame', 'end_frame', 'n_images', 'gt_tool_name', 'gt_tool_set_text', 'gt_is_evaluable']
    ].sort_values(['video_id', 'start_frame', 'end_frame']).reset_index(drop=True)
)


Audit-v11 clip eval items: 90


,example_uid,video_id,criterion,frame_range,start_frame,end_frame,n_images,gt_tool_name,gt_tool_set_text,gt_is_evaluable
0,00467596-8200-449c-8528-d4816ec2f6a2__C2__avg__c_000000_000150,00467596-8200-449c-8528-d4816ec2f6a2,C2,"[0, 150]",0,150,6,Hook,Hook,True
1,00467596-8200-449c-8528-d4816ec2f6a2__C2__avg__c_000450_001050,00467596-8200-449c-8528-d4816ec2f6a2,C2,"[450, 1050]",450,1050,21,Hook,Hook,True
2,00467596-8200-449c-8528-d4816ec2f6a2__C2__avg__c_001350_001650,00467596-8200-449c-8528-d4816ec2f6a2,C2,"[1350, 1650]",1350,1650,11,Hook,Hook,True
3,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg__c_001950_002400,00467596-8200-449c-8528-d4816ec2f6a2,C1,"[1950, 2400]",1950,2400,16,Hook,Hook,True
4,00467596-8200-449c-8528-d4816ec2f6a2__C2__avg__c_002100_002400,00467596-8200-449c-8528-d4816ec2f6a2,C2,"[2100, 2400]",2100,2400,11,Hook,Hook,True
...,...,...,...,...,...,...,...,...,...,...
85,27ea28e2-4ac8-45b4-a62d-3c4f22038188__C2__avg__c_000000_000750,27ea28e2-4ac8-45b4-a62d-3c4f22038188,C2,"[0, 750]",0,750,26,Hook,Hook,True
86,27ea28e2-4ac8-45b4-a62d-3c4f22038188__C1__avg__c_000300_000750,27ea28e2-4ac8-45b4-a62d-3c4f22038188,C1,"[300, 750]",300,750,16,Hook,Hook,True
87,27ea28e2-4ac8-45b4-a62d-3c4f22038188__C2__avg__c_001800_002100,27ea28e2-4ac8-45b4-a62d-3c4f22038188,C2,"[1800, 2100]",1800,2100,11,(absent),(absent),True
88,280aaeb3-c03c-4fe5-92ba-1e68ac5c2a45__C1__avg__c_000450_000900,280aaeb3-c03c-4fe5-92ba-1e68ac5c2a45,C1,"[450, 900]",450,900,16,Hook,Hook,True


In [4]:
PROMPT_VERSION = "2026-03-27-pm1s-b-v1"




PROMPT1_SYSTEM_PROMPT = "You identify the active working right-side laparoscopic instrument from a single frame. Return strict JSON only."
PROMPT2_ALL_IN_ONE_SYSTEM_PROMPT = "You answer surgical instrument questionnaires from images. Return strict JSON only."
PROMPT2_SEPARATE_SYSTEM_PROMPT = """You are a surgical instrument identification assistant.
You are viewing a frame from a laparoscopic cholecystectomy video.
Your task is to answer a visual question about the active working instrument on the right side of the image.
Reply with exactly one word:
- yes
- no
- uncertain
If you cannot clearly see evidence for either the yes features or the no features, reply uncertain.
Do not add any explanation or instrument name."""
REQUEST_TEMPERATURE = 0
REQUEST_ENABLE_THINKING = False


def compute_file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def build_prompt_signature(prompt_record):
    canonical = json.dumps(prompt_record, sort_keys=True, ensure_ascii=True)
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()


def build_text_prompt_record(text, label=None):
    record = {'type': 'text', 'text': text}
    if label:
        record['label'] = label
    return record


def build_image_prompt_record(path, label, reference_name=None, reference_description=None):
    record = {
        'type': 'image',
        'label': label,
        'path': str(Path(path)),
        'sha256': compute_file_sha256(path),
    }
    if reference_name is not None:
        record['reference_name'] = reference_name
    if reference_description is not None:
        record['reference_description'] = reference_description
    return record


def get_question_frame_paths(image_path: str):
    target_path = Path(image_path)
    try:
        frame_id = int(target_path.stem.split('_')[-1])
    except Exception:
        return [('target_frame', target_path)]

    frame_paths = [('target_frame', target_path)]
    before_path = target_path.with_name(f'frame_{frame_id - 30:06d}{target_path.suffix}')
    after_path = target_path.with_name(f'frame_{frame_id + 30:06d}{target_path.suffix}')
    if before_path.exists():
        frame_paths.append(('frame_minus_1s', before_path))
    if after_path.exists():
        frame_paths.append(('frame_plus_1s', after_path))
    return frame_paths


def build_question_frame_content(image_path: str):
    frame_paths = get_question_frame_paths(image_path)
    user_content = []
    user_record_content = []
    for label, frame_path in frame_paths:
        user_content.append({'type': 'image_url', 'image_url': {'url': image_path_to_data_url(Path(frame_path))}})
        user_record_content.append(build_image_prompt_record(frame_path, label=label))
    return {
        'frame_paths': [(label, str(path)) for label, path in frame_paths],
        'user_content': user_content,
        'user_record_content': user_record_content,
    }


def finalize_prompt_payload(method_key, system_prompt, user_content, user_record_content):
    prompt_record = {
        'method': method_key,
        'model': MODEL_ID,
        'temperature': REQUEST_TEMPERATURE,
        'enable_thinking': REQUEST_ENABLE_THINKING,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_record_content},
        ],
    }
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_content},
    ]
    return {'messages': messages, 'prompt_record': prompt_record}


def build_prompt1_direct_large_payload(image_path: str):
    tool_types_str = ', '.join(TOOL_TYPES)
    prompt = f"""
You are a surgical video analysis assistant.

You are viewing a single frame from a laparoscopic cholecystectomy video.
We want to determine whether the RIGHT instrument is actively performing the working action in this frame,
and if so, what tool it is.

CONTROLLED VOCABULARY for tool_type:
  [{tool_types_str}, Unknown, (not set)]

INSTRUMENT IDENTIFICATION NOTES:
- The RIGHT instrument is the active/working instrument in the surgical field.
- It is typically visible on the right side of the frame, performing the primary surgical action
  such as dissecting, cutting, cauterizing, or aspirating/irrigating.
- The working instrument is usually Hook, Maryland, Scissors, or Irrigator, but it may also be
  Grasper, Clipper, Unknown, or absent.
- Hook has a distinctive L-shaped or J-shaped hooked tip used for electrocautery dissection.
- Maryland is a bipolar dissector/forceps with two fine tapered jaws that open from a common pivot.
- Scissors have two blades that open and close for cutting.
- Irrigator is a smooth tubular suction-irrigation instrument, often with a hole-like opening.
- Grasper has two grasping jaws and is often broader/blunter than Maryland.
- Clipper has two jaws but they stay relatively parallel instead of opening like a fan from a pivot.

Your task:
1. Decide whether the active working RIGHT instrument is present and identifiable in this frame.
2. If yes, identify the tool type.

RULES:
- Use ONLY the controlled vocabulary for tool_type.
- If the working right instrument is not clearly present or cannot be identified, set right_present=false.
- If right_present is false, set right_tool_type to "(not set)".

Output format (STRICT JSON only; no markdown fences; no extra keys):
{{
  "right_present": true/false,
  "right_tool_type": "Grasper|Maryland|Hook|Irrigator|Scissors|Clipper|Unknown|(not set)",
  "confidence": 0.0,
  "reasoning": "brief explanation of what you see"
}}

CONSTRAINTS:
- confidence is a float in [0,1].
- reasoning should be 1-2 sentences max.
""".strip()

    user_content = [
        {'type': 'text', 'text': prompt},
        {'type': 'image_url', 'image_url': {'url': image_path_to_data_url(Path(image_path))}},
    ]
    user_record_content = [
        build_text_prompt_record(prompt, label='instruction'),
        build_image_prompt_record(image_path, label='target_frame'),
    ]
    return finalize_prompt_payload('prompt1_direct_large', PROMPT1_SYSTEM_PROMPT, user_content, user_record_content)
def normalize_tool_label(value):
    if value is None:
        return ''
    value = str(value).strip()
    if not value:
        return ''
    lowered = value.lower()
    mapping = {
        'grasper': 'Grasper',
        'maryland': 'Maryland',
        'hook': 'Hook',
        'irrigator': 'Irrigator',
        'scissors': 'Scissors',
        'clipper': 'Clipper',
        'unknown': 'Unknown',
        '(absent)': '(absent)',
        '(not set)': '(absent)',
        'not set': '(absent)',
        'none': '(absent)',
    }
    return mapping.get(lowered, value)


def parse_direct_prediction_json(raw_text):
    try:
        parsed = extract_json_object(raw_text)
    except Exception as exc:
        return {
            'raw': raw_text,
            'parse_error': str(exc),
            'right_present': False,
            'right_tool_type': '(absent)',
            'confidence': None,
            'reasoning': '',
            'predicted_tool': '(absent)',
        }

    right_present = bool(parsed.get('right_present', False))
    right_tool_type = normalize_tool_label(parsed.get('right_tool_type', '(absent)'))
    predicted_tool = right_tool_type if right_present else '(absent)'
    if predicted_tool == '(not set)':
        predicted_tool = '(absent)'
    return {
        'raw': raw_text,
        'parse_error': None,
        'right_present': right_present,
        'right_tool_type': right_tool_type,
        'confidence': parsed.get('confidence'),
        'reasoning': parsed.get('reasoning', ''),
        'predicted_tool': normalize_tool_label(predicted_tool),
    }


def ask_prompt1_direct_large(image_path: str):
    payload = build_prompt1_direct_large_payload(image_path)
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=payload['messages'],
        temperature=REQUEST_TEMPERATURE,
        extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
    )
    raw_text = response.choices[0].message.content
    return parse_direct_prediction_json(raw_text)


def _build_questionnaire_prompt(include_shape_examples=False):
    question_blocks = []
    for item in QUESTION_SPECS:
        block = f"- {item['qid']}: {item['question']}"
        if item.get('description'):
            block += f"\n  Yes definition: {item['description'].get('yes', '')}"
            block += f"\n  No definition: {item['description'].get('no', '')}"
            if item['description'].get('uncertain'):
                block += f"\n  Uncertain definition: {item['description'].get('uncertain', '')}"
        block += "\n  Allowed answers: Yes, No, Uncertain"
        question_blocks.append(block)

    prompt = f"""
You are a surgical instrument identification assistant.
You are viewing one or more frames from the same laparoscopic cholecystectomy clip.
The first image is the target frame. Additional images, when shown, are neighboring frames from about 1 second before or after.
Your task is to answer the exact questionnaire below about the actively working RIGHT-side instrument.

Rules:
- Answer every question with exactly one of: Yes, No, Uncertain.
- For the 10 instrument questions, judge the visible instrument morphology only.
- For the 3 frame-check questions, judge image visibility or instrument presence only.
- Use all provided frames together when they help resolve visibility or ambiguity.
- If the active working instrument is not clearly present, use Uncertain for morphology questions when needed.
- If you cannot clearly see evidence for either the Yes features or the No features, answer Uncertain.
- Do not skip any question.
""".strip()

    if include_shape_examples:
        prompt += "\n\nYou will also be shown 10 reference example images for the morphology questions. Use them as visual guidance for the shape concepts, then answer the questionnaire on the target frame and any provided neighboring frames."

    prompt += f"""

Questions:
{chr(10).join(question_blocks)}

Return STRICT JSON only with this schema:
{{
  "answers": {{
    "instrument_1": "Yes|No|Uncertain",
    "instrument_2": "Yes|No|Uncertain",
    "instrument_3": "Yes|No|Uncertain",
    "instrument_4": "Yes|No|Uncertain",
    "instrument_5": "Yes|No|Uncertain",
    "instrument_6": "Yes|No|Uncertain",
    "instrument_7": "Yes|No|Uncertain",
    "instrument_8": "Yes|No|Uncertain",
    "instrument_9": "Yes|No|Uncertain",
    "instrument_10": "Yes|No|Uncertain",
    "frame_check_1": "Yes|No|Uncertain",
    "frame_check_2": "Yes|No|Uncertain",
    "frame_check_3": "Yes|No|Uncertain"
  }},
  "brief_reasoning": "1-3 short sentences"
}}
"""
    return prompt.strip()


def build_prompt2_all_in_one_payload(image_path: str, include_shape_examples=False):
    questionnaire_prompt = _build_questionnaire_prompt(include_shape_examples=include_shape_examples)
    user_content = [{'type': 'text', 'text': questionnaire_prompt}]
    user_record_content = [build_text_prompt_record(questionnaire_prompt, label='questionnaire')]
    if include_shape_examples:
        for item in instrument_rubric:
            ref = reference_images[item['reference_name']]
            ref_text = f"Reference for instrument_{item['id']}: {item['question']}\nYes definition: {item['description'].get('yes', '')}\nNo definition: {item['description'].get('no', '')}"
            if item['description'].get('uncertain'):
                ref_text += f"\nUncertain definition: {item['description'].get('uncertain', '')}"
            user_content.append({'type': 'text', 'text': ref_text})
            user_content.append({'type': 'image_url', 'image_url': {'url': f"data:{ref['mime']};base64,{ref['b64']}"}})
            user_record_content.append(build_text_prompt_record(ref_text, label=f"reference_text_instrument_{item['id']}"))
            user_record_content.append(build_image_prompt_record(ref['path'], label=f"reference_image_instrument_{item['id']}", reference_name=item['reference_name'], reference_description=item['reference_description']))
    frame_bundle = build_question_frame_content(image_path)
    user_content.extend(frame_bundle['user_content'])
    user_record_content.extend(frame_bundle['user_record_content'])
    method_key = 'prompt2_all_in_one_json_with_shapes' if include_shape_examples else 'prompt2_all_in_one_json'
    payload = finalize_prompt_payload(method_key, PROMPT2_ALL_IN_ONE_SYSTEM_PROMPT, user_content, user_record_content)
    payload['prompt_record']['frames'] = frame_bundle['frame_paths']
    return payload


def build_prompt2_separate_call_payload(image_path: str, question_spec: dict, include_examples=True):
    user_content = []
    user_record_content = []
    if include_examples and question_spec['reference_name'] is not None:
        ref = reference_images[question_spec['reference_name']]
        ref_text = f"Reference example: {question_spec['reference_description']}"
        user_content.append({'type': 'text', 'text': ref_text})
        user_content.append({'type': 'image_url', 'image_url': {'url': f"data:{ref['mime']};base64,{ref['b64']}"}})
        user_record_content.append(build_text_prompt_record(ref_text, label='reference_text'))
        user_record_content.append(build_image_prompt_record(ref['path'], label='reference_image', reference_name=question_spec['reference_name'], reference_description=question_spec['reference_description']))

    guidance_text = f"Target image question: {question_spec['question']}"
    if question_spec.get('description'):
        guidance_text += (
            f"\nYes definition: {question_spec['description'].get('yes', '')}"
            f"\nNo definition: {question_spec['description'].get('no', '')}"
        )
        if question_spec['description'].get('uncertain'):
            guidance_text += f"\nUncertain definition: {question_spec['description'].get('uncertain', '')}"
    guidance_text += ' Use all provided frames together when they help resolve visibility or ambiguity. If you cannot clearly see evidence for either the yes features or the no features, answer uncertain. Answer with exactly one word: yes, no, or uncertain.'

    user_content.append({'type': 'text', 'text': guidance_text})
    user_record_content.append(build_text_prompt_record(guidance_text, label=f"question_{question_spec['qid']}"))
    frame_bundle = build_question_frame_content(image_path)
    user_content.extend(frame_bundle['user_content'])
    user_record_content.extend(frame_bundle['user_record_content'])
    method_key = 'prompt2_separate_calls' if include_examples else 'prompt2_separate_calls_no_examples'
    payload = finalize_prompt_payload(method_key, PROMPT2_SEPARATE_SYSTEM_PROMPT, user_content, user_record_content)
    payload['prompt_record']['question_id'] = question_spec['qid']
    payload['prompt_record']['frames'] = frame_bundle['frame_paths']
    return payload


def build_method_prompt_record(method_key, image_path: str):
    if method_key == 'prompt1_direct_large':
        return build_prompt1_direct_large_payload(image_path)['prompt_record']
    if method_key == 'prompt2_all_in_one_json':
        return build_prompt2_all_in_one_payload(image_path, include_shape_examples=False)['prompt_record']
    if method_key == 'prompt2_all_in_one_json_with_shapes':
        return build_prompt2_all_in_one_payload(image_path, include_shape_examples=True)['prompt_record']
    if method_key in {'prompt2_separate_calls', 'prompt2_separate_calls_no_examples'}:
        include_examples = method_key == 'prompt2_separate_calls'
        return {
            'method': method_key,
            'questions': {
                question_spec['qid']: build_prompt2_separate_call_payload(image_path, question_spec, include_examples=include_examples)['prompt_record']
                for question_spec in QUESTION_SPECS
            },
        }
    raise KeyError(f'Unknown method key: {method_key}')


def ask_prompt2_all_in_one_json(image_path: str, include_shape_examples=False):
    payload = build_prompt2_all_in_one_payload(image_path, include_shape_examples=include_shape_examples)
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=payload['messages'],
        temperature=REQUEST_TEMPERATURE,
        extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
    )
    raw_text = response.choices[0].message.content
    parsed = extract_json_object(raw_text)
    answers = {
        q['qid']: normalize_answer(parsed.get('answers', {}).get(q['qid'], 'Uncertain'))
        for q in QUESTION_SPECS
    }
    return {'raw': raw_text, 'answers': answers, 'aggregation': aggregate_tool_prediction(answers)}


def ask_prompt2_separate_calls(image_path: str, question_spec: dict, include_examples=True):
    payload = build_prompt2_separate_call_payload(image_path, question_spec, include_examples=include_examples)
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=payload['messages'],
        temperature=REQUEST_TEMPERATURE,
        extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
    )
    raw_text = response.choices[0].message.content
    return {'raw': raw_text, 'answer': extract_binary_answer(raw_text)}


def compute_normalized_tool_scores(answer_map):
    tool_breakdown = {}
    for tool in TOOL_TYPES:
        scored_items = []
        earned_points = 0
        total_possible = 0
        for item in instrument_rubric:
            expected, points = item['scores'].get(tool, (None, 0))
            if expected is None or points <= 0:
                continue
            total_possible += points
            observed = normalize_answer(answer_map.get(f"instrument_{item['id']}", 'Uncertain'))
            expected = normalize_answer(expected)
            matched = observed == expected
            if matched:
                earned_points += points
            scored_items.append({
                'question_id': f"instrument_{item['id']}",
                'expected': expected,
                'observed': observed,
                'points': points,
                'matched': matched,
                'earned': points if matched else 0,
            })
        normalized_score = (earned_points / total_possible) if total_possible else 0.0
        tool_breakdown[tool] = {
            'earned_points': earned_points,
            'total_possible': total_possible,
            'normalized_score': normalized_score,
            'items': scored_items,
        }
    sorted_tools = sorted(
        ((tool, info['normalized_score']) for tool, info in tool_breakdown.items()),
        key=lambda x: x[1],
        reverse=True,
    )
    predicted_tool = sorted_tools[0][0] if sorted_tools else 'Unknown'
    return {
        'tool_breakdown': tool_breakdown,
        'tool_scores': {tool: info['normalized_score'] for tool, info in tool_breakdown.items()},
        'sorted_tools': sorted_tools,
        'predicted_tool': predicted_tool,
    }


def aggregate_tool_prediction(answer_map):
    present_answer = normalize_answer(answer_map.get('frame_check_1', 'Uncertain'))
    if present_answer != 'Yes':
        return {'predicted_tool': '(absent)', 'tool_scores': {}, 'sorted_tools': [], 'tool_breakdown': {}}
    return compute_normalized_tool_scores(answer_map)


PROMPT_VERSION = "2026-05-23-audit-v11-right-tool-rubric-qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps"

METHOD_SPECS = {
    'prompt2_all_in_one_json': {
        'label': 'Prompt 2A: all questions in one JSON',
        'produces_questions': True,
    },
    'prompt2_all_in_one_json_with_shapes': {
        'label': 'Prompt 2B: all questions in one JSON + 10 shape examples',
        'produces_questions': True,
    },
    'prompt2_separate_calls': {
        'label': 'Prompt 2C: separate per-question calls with examples',
        'produces_questions': True,
    },
    'prompt2_separate_calls_no_examples': {
        'label': 'Prompt 2D: separate per-question calls without examples',
        'produces_questions': True,
    },
}
METHOD_ORDER = list(METHOD_SPECS.keys())
QUESTION_LOOKUP = {q['qid']: q for q in QUESTION_SPECS}


def get_question_frame_paths(image_path):
    if isinstance(image_path, (list, tuple)):
        frame_paths = []
        for idx, item in enumerate(image_path):
            frame_paths.append((f'clip_frame_{idx:03d}', Path(item)))
        return frame_paths

    target_path = Path(image_path)
    return [('target_frame', target_path)]


In [ ]:

from tqdm.auto import tqdm


def load_cache():
    if CACHE_PATH.is_file():
        return json.loads(CACHE_PATH.read_text())
    return {}


def save_cache(cache):
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_text(json.dumps(cache, indent=2))


def cache_entry_is_current(ex_cache, method_key, image_path):
    entry = ex_cache.get(method_key)
    if not isinstance(entry, dict):
        return False
    if entry.get('prompt_version') != PROMPT_VERSION:
        return False
    expected_prompt_record = build_method_prompt_record(method_key, image_path)
    expected_signature = build_prompt_signature(expected_prompt_record)
    stored_signature = entry.get('prompt_signature')
    if stored_signature is None and isinstance(entry.get('prompt_record'), dict):
        stored_signature = build_prompt_signature(entry['prompt_record'])
    return stored_signature == expected_signature


def attach_prompt_cache_metadata(result, method_key, image_path):
    prompt_record = build_method_prompt_record(method_key, image_path)
    result['prompt_version'] = PROMPT_VERSION
    result['prompt_record'] = prompt_record
    result['prompt_signature'] = build_prompt_signature(prompt_record)
    return result


def get_example_gt_tool(ex):
    return normalize_tool_label(ex.get('gt_tool_name', ''))


def get_method_pred_tool(ex_cache, method_key):
    if method_key == 'prompt1_direct_large':
        return normalize_tool_label(ex_cache.get(method_key, {}).get('predicted_tool', ''))
    return normalize_tool_label(ex_cache.get(method_key, {}).get('aggregation', {}).get('predicted_tool', ''))


def compute_running_accuracy(cache_dict, processed_examples):
    rows = []
    for method_key in METHOD_ORDER:
        total = 0
        correct = 0
        for ex in processed_examples:
            ex_cache = cache_dict.get(ex['example_uid'], {})
            if not cache_entry_is_current(ex_cache, method_key, ex['image_path']):
                continue
            gt_tool = get_example_gt_tool(ex)
            pred_tool = get_method_pred_tool(ex_cache, method_key)
            if not gt_tool or not pred_tool:
                continue
            total += 1
            correct += int(gt_tool == pred_tool)
        acc = (correct / total) if total else None
        rows.append((method_key, correct, total, acc))
    return rows


cache = load_cache()
print('Cached examples:', len(cache))
print('Prompt version:', PROMPT_VERSION)

MAX_EXAMPLES = None  # set an int for a smaller test run
examples_to_run = eval_examples[:MAX_EXAMPLES] if MAX_EXAMPLES else eval_examples
processed_examples = []

progress = tqdm(examples_to_run, total=len(examples_to_run), desc='Running frame inference', dynamic_ncols=True)
for idx, ex in enumerate(progress, start=1):
    ex_key = ex['example_uid']
    progress.set_postfix_str(ex_key)
    print(f"[{idx}/{len(examples_to_run)}] {ex_key} | {ex['n_images']} images | frames {ex['frame_range']}")
    ex_cache = cache.get(ex_key, {})

    if not cache_entry_is_current(ex_cache, 'prompt2_all_in_one_json', ex['image_path']):
        result = ask_prompt2_all_in_one_json(ex['image_path'], include_shape_examples=False)
        result = attach_prompt_cache_metadata(result, 'prompt2_all_in_one_json', ex['image_path'])
        ex_cache['prompt2_all_in_one_json'] = result
        cache[ex_key] = ex_cache
        save_cache(cache)

    if not cache_entry_is_current(ex_cache, 'prompt2_all_in_one_json_with_shapes', ex['image_path']):
        result = ask_prompt2_all_in_one_json(ex['image_path'], include_shape_examples=True)
        result = attach_prompt_cache_metadata(result, 'prompt2_all_in_one_json_with_shapes', ex['image_path'])
        ex_cache['prompt2_all_in_one_json_with_shapes'] = result
        cache[ex_key] = ex_cache
        save_cache(cache)

    if not cache_entry_is_current(ex_cache, 'prompt2_separate_calls', ex['image_path']):
        prompt2_answers = {}
        prompt2_raw = {}
        for question_spec in QUESTION_SPECS:
            result = ask_prompt2_separate_calls(ex['image_path'], question_spec, include_examples=True)
            prompt2_answers[question_spec['qid']] = result['answer']
            prompt2_raw[question_spec['qid']] = result['raw']
        ex_cache['prompt2_separate_calls'] = attach_prompt_cache_metadata({
            'answers': prompt2_answers,
            'raw': prompt2_raw,
            'aggregation': aggregate_tool_prediction(prompt2_answers),
        }, 'prompt2_separate_calls', ex['image_path'])
        cache[ex_key] = ex_cache
        save_cache(cache)

    if not cache_entry_is_current(ex_cache, 'prompt2_separate_calls_no_examples', ex['image_path']):
        prompt2_answers = {}
        prompt2_raw = {}
        for question_spec in QUESTION_SPECS:
            result = ask_prompt2_separate_calls(ex['image_path'], question_spec, include_examples=False)
            prompt2_answers[question_spec['qid']] = result['answer']
            prompt2_raw[question_spec['qid']] = result['raw']
        ex_cache['prompt2_separate_calls_no_examples'] = attach_prompt_cache_metadata({
            'answers': prompt2_answers,
            'raw': prompt2_raw,
            'aggregation': aggregate_tool_prediction(prompt2_answers),
        }, 'prompt2_separate_calls_no_examples', ex['image_path'])
        cache[ex_key] = ex_cache
        save_cache(cache)

    cache[ex_key] = ex_cache
    processed_examples.append(ex)

    running_rows = compute_running_accuracy(cache, processed_examples)
    brief = ' | '.join(
        f"{METHOD_SPECS[m]['label'] if isinstance(METHOD_SPECS[m], dict) else METHOD_SPECS[m]}: {c}/{t} ({a:.3f})"
        for m, c, t, a in running_rows if t
    )
    if brief:
        print('  Running aggregation acc:', brief)

print('Finished inference. Cache saved to', CACHE_PATH)


Cached examples: 0
Prompt version: 2026-05-23-audit-v11-right-tool-rubric-qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps


Running frame inference:   0%|                                                                  | 0/90 [00:00<…

[1/90] 00467596-8200-449c-8528-d4816ec2f6a2__C1__avg__c_001950_002400 | 16 images | frames [1950, 2400]
  Running aggregation acc: Prompt 2A: all questions in one JSON: 0/1 (0.000) | Prompt 2B: all questions in one JSON + 10 shape examples: 0/1 (0.000) | Prompt 2C: separate per-question calls with examples: 0/1 (0.000) | Prompt 2D: separate per-question calls without examples: 0/1 (0.000)
[2/90] 00467596-8200-449c-8528-d4816ec2f6a2__C2__avg__c_000000_000150 | 6 images | frames [0, 150]
  Running aggregation acc: Prompt 2A: all questions in one JSON: 0/2 (0.000) | Prompt 2B: all questions in one JSON + 10 shape examples: 1/2 (0.500) | Prompt 2C: separate per-question calls with examples: 0/2 (0.000) | Prompt 2D: separate per-question calls without examples: 0/2 (0.000)
[3/90] 00467596-8200-449c-8528-d4816ec2f6a2__C2__avg__c_000450_001050 | 21 images | frames [450, 1050]
  Running aggregation acc: Prompt 2A: all questions in one JSON: 0/3 (0.000) | Prompt 2B: all questions in one JSON + 

In [9]:


def evaluate_results(cache_dict):
    question_pairs = {method: defaultdict(list) for method, spec in METHOD_SPECS.items() if spec['produces_questions']}
    clip_question_scores = {method: defaultdict(lambda: defaultdict(list)) for method, spec in METHOD_SPECS.items() if spec['produces_questions']}
    aggregation_pairs = {method: [] for method in METHOD_ORDER}
    clip_aggregation_scores = {method: defaultdict(list) for method in METHOD_ORDER}
    clip_aggregation_matches = {method: defaultdict(list) for method in METHOD_ORDER}
    detailed_rows = []

    for ex in eval_examples:
        ex_key = ex['example_uid']
        if ex_key not in cache_dict:
            continue
        clip_uid = ex['clip_uid']
        ex_cache = cache_dict[ex_key]
        gt_tool = normalize_tool_label(ex['gt_tool_name'])

        prompt_outputs = {
            'prompt1_direct_large': {
                'answers': None,
                'predicted_tool': normalize_tool_label(ex_cache.get('prompt1_direct_large', {}).get('predicted_tool', '')),
            },
            'prompt2_all_in_one_json': {
                'answers': ex_cache.get('prompt2_all_in_one_json', {}).get('answers', {}),
                'predicted_tool': normalize_tool_label(ex_cache.get('prompt2_all_in_one_json', {}).get('aggregation', {}).get('predicted_tool', '')),
            },
            'prompt2_all_in_one_json_with_shapes': {
                'answers': ex_cache.get('prompt2_all_in_one_json_with_shapes', {}).get('answers', {}),
                'predicted_tool': normalize_tool_label(ex_cache.get('prompt2_all_in_one_json_with_shapes', {}).get('aggregation', {}).get('predicted_tool', '')),
            },
            'prompt2_separate_calls': {
                'answers': ex_cache.get('prompt2_separate_calls', {}).get('answers', {}),
                'predicted_tool': normalize_tool_label(ex_cache.get('prompt2_separate_calls', {}).get('aggregation', {}).get('predicted_tool', '')),
            },
            'prompt2_separate_calls_no_examples': {
                'answers': ex_cache.get('prompt2_separate_calls_no_examples', {}).get('answers', {}),
                'predicted_tool': normalize_tool_label(ex_cache.get('prompt2_separate_calls_no_examples', {}).get('aggregation', {}).get('predicted_tool', '')),
            },
        }

        for method, output in prompt_outputs.items():
            pred_answers = output['answers']
            if pred_answers:
                for q in QUESTION_SPECS:
                    gt = normalize_answer(ex['gt_answers'].get(q['qid'], ''))
                    if not gt:
                        continue
                    pred = normalize_answer(pred_answers.get(q['qid'], 'Uncertain'))
                    correct = int(pred == gt)
                    question_pairs[method][q['qid']].append((pred, gt))
                    clip_question_scores[method][q['qid']][clip_uid].append(correct)
                    detailed_rows.append({
                        'example_uid': ex_key,
                        'clip_uid': clip_uid,
                        'frame_tag': ex['frame_tag'],
                        'frame_id': ex['frame_id'],
                        'method': method,
                        'target_type': 'question',
                        'question_id': q['qid'],
                        'question_text': q['question'],
                        'gt': gt,
                        'pred': pred,
                        'correct': correct,
                    })

            pred_tool = normalize_tool_label(output['predicted_tool'])
            if gt_tool and pred_tool:
                correct = int(pred_tool == gt_tool)
                aggregation_pairs[method].append((pred_tool, gt_tool))
                clip_aggregation_scores[method][clip_uid].append(correct)
                clip_aggregation_matches[method][clip_uid].append(correct)
                detailed_rows.append({
                    'example_uid': ex_key,
                    'clip_uid': clip_uid,
                    'frame_tag': ex['frame_tag'],
                    'frame_id': ex['frame_id'],
                    'method': method,
                    'target_type': 'aggregation',
                    'question_id': 'tool_name',
                    'question_text': 'Final tool prediction',
                    'gt': gt_tool,
                    'pred': pred_tool,
                    'correct': correct,
                })

    metric_rows = []
    for method in METHOD_ORDER:
        if METHOD_SPECS[method]['produces_questions']:
            for q in QUESTION_SPECS:
                pairs = question_pairs[method][q['qid']]
                if not pairs:
                    continue
                micro = sum(int(pred == gt) for pred, gt in pairs) / len(pairs)
                per_clip = [sum(vals) / len(vals) for vals in clip_question_scores[method][q['qid']].values() if vals]
                macro = sum(per_clip) / len(per_clip) if per_clip else None
                metric_rows.append({
                    'method': method,
                    'target_type': 'question',
                    'target_id': q['qid'],
                    'question_text': q['question'],
                    'n': len(pairs),
                    'micro_accuracy': micro,
                    'macro_accuracy': macro,
                })

        agg_pairs = aggregation_pairs[method]
        if agg_pairs:
            micro = sum(int(pred == gt) for pred, gt in agg_pairs) / len(agg_pairs)
            per_clip = [sum(vals) / len(vals) for vals in clip_aggregation_scores[method].values() if vals]
            macro = sum(per_clip) / len(per_clip) if per_clip else None
            metric_rows.append({
                'method': method,
                'target_type': 'aggregation',
                'target_id': 'tool_name',
                'question_text': 'Final tool prediction',
                'n': len(agg_pairs),
                'micro_accuracy': micro,
                'macro_accuracy': macro,
            })

        pass_by_clip = [int(any(vals)) for vals in clip_aggregation_matches[method].values() if vals]
        if pass_by_clip:
            pass_at_3 = sum(pass_by_clip) / len(pass_by_clip)
            metric_rows.append({
                'method': method,
                'target_type': 'pass_at_3',
                'target_id': 'tool_name',
                'question_text': 'Any of start/mid/end predicts the correct tool',
                'n': len(pass_by_clip),
                'micro_accuracy': pass_at_3,
                'macro_accuracy': pass_at_3,
            })

    metric_columns = ['method', 'target_type', 'target_id', 'question_text', 'n', 'micro_accuracy', 'macro_accuracy']
    detail_columns = ['example_uid', 'clip_uid', 'frame_tag', 'frame_id', 'method', 'target_type', 'question_id', 'question_text', 'gt', 'pred', 'correct']
    metrics_df = pd.DataFrame(metric_rows, columns=metric_columns)
    detailed_df = pd.DataFrame(detailed_rows, columns=detail_columns)
    return metrics_df, detailed_df


metrics_df, detailed_df = evaluate_results(cache)
print('Metric rows:', len(metrics_df))
print('Detailed rows:', len(detailed_df))
if metrics_df.empty:
    print('No metrics available yet. Run the inference cell to populate the cache with the new prompt outputs.')
else:
    display(metrics_df.sort_values(['target_type', 'target_id', 'method']).reset_index(drop=True))


Metric rows: 8
Detailed rows: 352


,method,target_type,target_id,question_text,n,micro_accuracy,macro_accuracy
0,prompt2_all_in_one_json,aggregation,tool_name,Final tool prediction,88,0.397727,0.397727
1,prompt2_all_in_one_json_with_shapes,aggregation,tool_name,Final tool prediction,88,0.568182,0.568182
2,prompt2_separate_calls,aggregation,tool_name,Final tool prediction,88,0.409091,0.409091
3,prompt2_separate_calls_no_examples,aggregation,tool_name,Final tool prediction,88,0.386364,0.386364
4,prompt2_all_in_one_json,pass_at_3,tool_name,Any of start/mid/end predicts the correct tool,88,0.397727,0.397727
5,prompt2_all_in_one_json_with_shapes,pass_at_3,tool_name,Any of start/mid/end predicts the correct tool,88,0.568182,0.568182
6,prompt2_separate_calls,pass_at_3,tool_name,Any of start/mid/end predicts the correct tool,88,0.409091,0.409091
7,prompt2_separate_calls_no_examples,pass_at_3,tool_name,Any of start/mid/end predicts the correct tool,88,0.386364,0.386364


In [10]:

question_metrics = metrics_df[metrics_df['target_type'] == 'question'].copy()
aggregation_metrics = metrics_df[metrics_df['target_type'] == 'aggregation'].copy()
pass_at_3_metrics = metrics_df[metrics_df['target_type'] == 'pass_at_3'].copy()

print('Question Accuracy Summary')
if question_metrics.empty:
    print('No question-level metrics available yet.')
else:
    question_summary = question_metrics[['target_id', 'question_text', 'method', 'micro_accuracy', 'macro_accuracy', 'n']].copy()
    display(question_summary.sort_values(['target_id', 'method']).reset_index(drop=True))

print('Aggregation Accuracy Summary')
if aggregation_metrics.empty:
    print('No aggregation metrics available yet.')
else:
    display(aggregation_metrics[['method', 'target_id', 'n', 'micro_accuracy', 'macro_accuracy']].sort_values(['method']).reset_index(drop=True))

print('Pass@3 Summary')
if pass_at_3_metrics.empty:
    print('No pass@3 metrics available yet.')
else:
    display(pass_at_3_metrics[['method', 'n', 'micro_accuracy', 'macro_accuracy']].rename(columns={'micro_accuracy': 'pass@3', 'macro_accuracy': 'pass@3_macro'}).sort_values(['method']).reset_index(drop=True))

print('Overall method summary')
overall_rows = []
for method in METHOD_ORDER:
    subset = detailed_df[detailed_df['method'] == method]
    q_subset = subset[subset['target_type'] == 'question']
    a_subset = subset[subset['target_type'] == 'aggregation']
    method_question_metrics = question_metrics[question_metrics['method'] == method]
    method_agg_metrics = aggregation_metrics[aggregation_metrics['method'] == method]
    method_pass_metrics = pass_at_3_metrics[pass_at_3_metrics['method'] == method]
    overall_rows.append({
        'method': method,
        'label': METHOD_SPECS[method]['label'],
        'question_micro_accuracy': q_subset['correct'].mean() if len(q_subset) else None,
        'question_macro_accuracy': method_question_metrics['macro_accuracy'].mean() if len(method_question_metrics) else None,
        'aggregation_micro_accuracy': a_subset['correct'].mean() if len(a_subset) else None,
        'aggregation_macro_accuracy': method_agg_metrics['macro_accuracy'].iloc[0] if len(method_agg_metrics) else None,
        'pass@3': method_pass_metrics['micro_accuracy'].iloc[0] if len(method_pass_metrics) else None,
    })

display(pd.DataFrame(overall_rows))


Question Accuracy Summary
No question-level metrics available yet.
Aggregation Accuracy Summary


,method,target_id,n,micro_accuracy,macro_accuracy
0,prompt2_all_in_one_json,tool_name,88,0.397727,0.397727
1,prompt2_all_in_one_json_with_shapes,tool_name,88,0.568182,0.568182
2,prompt2_separate_calls,tool_name,88,0.409091,0.409091
3,prompt2_separate_calls_no_examples,tool_name,88,0.386364,0.386364


Pass@3 Summary


,method,n,pass@3,pass@3_macro
0,prompt2_all_in_one_json,88,0.397727,0.397727
1,prompt2_all_in_one_json_with_shapes,88,0.568182,0.568182
2,prompt2_separate_calls,88,0.409091,0.409091
3,prompt2_separate_calls_no_examples,88,0.386364,0.386364


Overall method summary


,method,label,question_micro_accuracy,question_macro_accuracy,aggregation_micro_accuracy,aggregation_macro_accuracy,pass@3
0,prompt2_all_in_one_json,Prompt 2A: all questions in one JSON,None,None,0.397727,0.397727,0.397727
1,prompt2_all_in_one_json_with_shapes,Prompt 2B: all questions in one JSON + 10 shape examples,None,None,0.568182,0.568182,0.568182
2,prompt2_separate_calls,Prompt 2C: separate per-question calls with examples,None,None,0.409091,0.409091,0.409091
3,prompt2_separate_calls_no_examples,Prompt 2D: separate per-question calls without examples,None,None,0.386364,0.386364,0.386364


In [11]:

def clip_tool_from_segments(segments):
    tools = sorted({
        normalize_tool_label(seg.get('tool_type'))
        for seg in segments
        if normalize_tool_label(seg.get('tool_type'))
    })
    if not tools:
        return '(absent)', '(absent)'
    if len(tools) == 1:
        return tools[0], tools[0]
    return '(multi)', ' | '.join(tools)


def compare_with_cvsctx_v3(results_path, eval_examples):
    compare_rows = json.loads(Path(results_path).read_text()).get('results', [])
    baseline = {}
    for row in compare_rows:
        if row.get('status') != 'ok' or row.get('method') != 'structured_prediction_cvs_context':
            continue
        label, label_text = clip_tool_from_segments(row.get('extracted_actions', {}).get('right', []))
        baseline[row['example_id']] = {
            'pred_tool_name': label,
            'pred_tool_set_text': label_text,
        }

    evaluable = [ex for ex in eval_examples if ex.get('gt_tool_name')]
    rows = []

    for method in METHOD_ORDER:
        total = 0
        available = 0
        correct = 0
        for ex in evaluable:
            total += 1
            payload = cache.get(ex['example_uid'], {}).get(method, {})
            pred_tool = normalize_tool_label((payload.get('aggregation') or {}).get('predicted_tool', ''))
            if not pred_tool:
                continue
            available += 1
            correct += int(pred_tool == ex['gt_tool_name'])
        rows.append({
            'model_family': 'rubric_tool',
            'method': method,
            'label': METHOD_SPECS[method]['label'],
            'n_evaluable_clips': total,
            'n_available_predictions': available,
            'coverage': (available / total) if total else None,
            'accuracy_on_available': (correct / available) if available else None,
            'accuracy_missing_as_wrong': (correct / total) if total else None,
        })

    total = 0
    available = 0
    correct = 0
    baseline_detail = []
    for ex in evaluable:
        total += 1
        item = baseline.get(ex['example_uid'])
        if item is None:
            continue
        available += 1
        is_correct = int(item['pred_tool_name'] == ex['gt_tool_name'])
        correct += is_correct
        baseline_detail.append({
            'example_uid': ex['example_uid'],
            'criterion': ex['criterion'],
            'gt_tool_name': ex['gt_tool_name'],
            'baseline_pred_tool_name': item['pred_tool_name'],
            'baseline_pred_tool_set_text': item['pred_tool_set_text'],
            'correct': is_correct,
        })

    rows.append({
        'model_family': 'cvsctx_v3',
        'method': 'structured_prediction_cvs_context',
        'label': 'CVS-context v3 structured_prediction_cvs_context',
        'n_evaluable_clips': total,
        'n_available_predictions': available,
        'coverage': (available / total) if total else None,
        'accuracy_on_available': (correct / available) if available else None,
        'accuracy_missing_as_wrong': (correct / total) if total else None,
    })

    return pd.DataFrame(rows), pd.DataFrame(baseline_detail)


comparison_df, baseline_detail_df = compare_with_cvsctx_v3(COMPARE_RESULTS_PATH, eval_examples)
print('Clip-level right-tool comparison against cvsctx_v3 results')
display(comparison_df.sort_values(['model_family', 'method']).reset_index(drop=True))

if not baseline_detail_df.empty:
    print('Baseline cvsctx_v3 per-clip tool comparison details')
    display(baseline_detail_df.sort_values(['criterion', 'example_uid']).reset_index(drop=True))


Clip-level right-tool comparison against cvsctx_v3 results


,model_family,method,label,n_evaluable_clips,n_available_predictions,coverage,accuracy_on_available,accuracy_missing_as_wrong
0,cvsctx_v3,structured_prediction_cvs_context,CVS-context v3 structured_prediction_cvs_context,88,88,1.0,0.590909,0.590909
1,rubric_tool,prompt2_all_in_one_json,Prompt 2A: all questions in one JSON,88,88,1.0,0.397727,0.397727
2,rubric_tool,prompt2_all_in_one_json_with_shapes,Prompt 2B: all questions in one JSON + 10 shape examples,88,88,1.0,0.568182,0.568182
3,rubric_tool,prompt2_separate_calls,Prompt 2C: separate per-question calls with examples,88,88,1.0,0.409091,0.409091
4,rubric_tool,prompt2_separate_calls_no_examples,Prompt 2D: separate per-question calls without examples,88,88,1.0,0.386364,0.386364


Baseline cvsctx_v3 per-clip tool comparison details


,example_uid,criterion,gt_tool_name,baseline_pred_tool_name,baseline_pred_tool_set_text,correct
0,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg__c_001950_002400,C1,Hook,Hook,Hook,1
1,004fa282-7e76-473e-bf34-01fe831368dc__C1__avg__c_001650_001950,C1,Hook,Hook,Hook,1
2,0405e4b0-843a-469d-a235-6a12383bc1db__C1__avg__c_001500_002100,C1,Hook,Hook,Hook,1
3,0f7bd02b-4292-432a-a1bc-bfdf3b53d56b__C1__avg__c_001800_002250,C1,Hook,Hook,Hook,1
4,144fef8d-eaed-48ac-ba3b-a7a214e95f42__C1__avg__c_001050_001500,C1,Hook,Hook,Hook,1
...,...,...,...,...,...,...
83,1ff8fc19-8a7f-42e1-b8a4-30a25194c981__C3__avg__c_000450_000900,C3,Maryland,Hook,Hook,0
84,1ff8fc19-8a7f-42e1-b8a4-30a25194c981__C3__avg__c_001200_001650,C3,Maryland,(absent),(absent),0
85,215b09a5-a33d-45d7-8cb0-a7be8474ff01__C3__avg__c_001350_001800,C3,(absent),(absent),(absent),1
86,23643717-6c56-49ef-8761-29b64ca47a6d__C3__avg__c_000000_000300,C3,Maryland,(absent),(absent),0
